## Direct Preference Alignment with Direct Preference Optimization (DPO)

This notebook demonstrates example of fine-tuning a language model using Direct Preference Optimization (DPO) using SmolLM2-135M-Instruct model which has already been SFT trained, so it is compatible with DPO

### Import required modules

In [ ]:
import torch
import transformers, datasets, trl
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig

In [21]:
print("transformers version:", transformers.__version__)
print("torch version:", torch.__version__)
print("datasets version:", datasets.__version__)
print("trl version:", trl.__version__)

transformers version: 5.15.1
torch version: 2.13.0+cu130
datasets version: 5.0.1
trl version: 1.10.0


### load and format dataset

In [ ]:
# download and load dataset

dataset = load_dataset(path="trl-lib/ultrafeedback_binarized", split="train")

Generating test split: 100%|██████████| 1000/1000 [00:00<00:00, 33092.72 examples/s]


In [45]:
dataset

Dataset({
    features: ['chosen', 'rejected', 'score_chosen', 'score_rejected'],
    num_rows: 62135
})

In [33]:
# View sample dataset
dataset[100]

{'chosen': [{'content': 'You will be given a definition of a task first, then some input of the task.\nIn this task, you are given two questions about a domain. Your task is to combine the main subjects of the questions to write a new, natural-sounding question. For example, if the first question is about the tallness of the president and the second question is about his performance at college, the new question can be about his tallness at college. Try to find the main idea of each question, then combine them; you can use different words or make the subjects negative (i.e., ask about shortness instead of tallness) to combine the subjects. The questions are in three domains: presidents, national parks, and dogs. Each question has a keyword indicating its domain. Keywords are "this national park", "this dog breed", and "this president", which will be replaced with the name of an actual president, a national park, or a breed of dog. Hence, in the new question, this keyword should also be 

dataset is in conversation format with alternating user and assistant roles. Dataset also contains both chosen and rejected samples to teach the model to align preference to the chosen sample.

### Select and load model

In [ ]:
model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map="auto", 
    torch_dtype=torch.float32).to(device)

# disable attention keys and values caching for DPO training
# During DPO fine-tuning, however, caching is generally unnecessary 
# and can conflict with gradient checkpointing or consume extra GPU memory
model.config.use_cache = False

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# configure tokenizer padding token to be the same as the end of sequence token
tokenizer.pad_token = tokenizer.eos_token

# set name for the finetuned model to be saved to
output_dir = "ft-models/SmolLM2-135M-Instruct-DPO"
finetuned_model_name = "SmolLM2-135M-Instruct-DPO"
finetune_tags = ['smoll-lm', 'dpo', 'preference-alignment']


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 954.33it/s] 


### View the architecture of the model

In [10]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576, padding_idx=2)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (r

Training batches contain sequences of different lengths, so shorter sequences must be padded to match the longest sequence. Some causal language models do not define a dedicated padding token; the tokenizer padding line supplies one by reusing the existing EOS token.

The trainer’s attention mask should ensure that padded positions are ignored when calculating attention and loss. This assignment changes the tokenizer’s configuration; it does not resize the model vocabulary or create a new token.

In [24]:
traininig_args = DPOConfig(
    # training batch size per GPU
    per_device_train_batch_size=4,
    # Number of updates steps to accumulate before performing a backward/update pass.
    # Effective batch size = per_device_train_batch_size * gradient_accumulation_steps
    gradient_accumulation_steps=4,
    # Save memory by not storing activation during forward pass
    # instead recompute them during the backward pass
    gradient_checkpointing=True,
    # learning rate
    learning_rate=1e-5,
    # learning rate schedule - 'cosine' gradually decreases the learning rate following a cosine curve
    # other options include 'linear', 'constant', 'constant_with_warmup', 'polynomial', 'cosine_with_restarts'
    lr_scheduler_type='cosine',
    # Total number of training steps to perform. If provided, overrides num_train_epochs.
    max_steps=200,
    # disable model checkpointing during training
    save_strategy='no',
    # how often to log training metrics. Set to 'steps' to log every logging_steps, or 'epoch' to log at the end of each epoch.
    logging_steps = 1,
    # Directory to save model outputs and checkpoints.
    output_dir=output_dir,
    # Number of steps for learning rate warmup. During the warmup phase, 
    # the learning rate increases linearly from 0 to the initial learning rate set in the optimizer.
    # This can help stabilize training in the early stages.
    warmup_steps=20,
    # Use bfloat16 precision for faster training
    bf16=True,
    # Disable wandb/tensorboard logging
    report_to="none",
    # Keep all columns in dataset even if not used by the model. This is useful for debugging or when you want to keep additional information in the dataset.
    remove_unused_columns=False,
    # DPO-specific temperature parameter. This controls the strength of the preference model
    # Lower values (like 0.1) make the model more confident and conservative in its preferences, while higher values (like 1.0) make it more uncertain.
    beta=0.1,
    # Maximum length of prompt's + response in tokens. This is the total length of the input and output combined.
    max_length=1536,
)

### Build trainer from DPOTrainer

In [25]:
trainer = DPOTrainer(
    # The model to be trained. This should be a causal language model.
    model=model,
    # Training configuration parameters. This includes settings like batch size, learning rate, and number of training steps.
    args=traininig_args,
    # Dataset containing preferred/rejected reponse pairs. Each entry should have a 'query', 'response', and 'score' field.
    train_dataset=dataset,
    # Tokenizer used for encoding the dataset. This should match the tokenizer used for the model
    processing_class=tokenizer,
)

Dropping fully truncated examples from train dataset: 100%|██████████| 62135/62135 [00:02<00:00, 28470.50 examples/s]
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 854.56it/s]


In [26]:
# Train the model using DPO
trainer.train()

# Save the finetuned model to the specified output directory
trainer.save_model(output_dir)

[W825 13:23:04.812735958 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 2359296000 bytes (free: 1995964416, total: 16757751808).


Step,Training Loss
1,0.693147
2,0.693147
3,0.716142
4,0.693861
5,0.702671
6,0.680308
7,0.696462
8,0.687800
9,0.724063
10,0.688006


[W825 13:23:13.039055607 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 2415919104 bytes (free: 1066926080, total: 16757751808).
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.11it/s]


In [39]:
eval_dataset = load_dataset(path="trl-lib/ultrafeedback_binarized", split="test").select([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [40]:
# Evaluate the finetuned model (optional)
results = trainer.evaluate(eval_dataset=eval_dataset)

Tokenizing eval dataset: 100%|██████████| 10/10 [00:00<00:00, 393.34 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 10/10 [00:00<00:00, 3964.00 examples/s]


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.601094,0.688808,200,1.403481,3051683.000000,3.143628,3.413518,0.636028,-0.072777,-0.250785,0.708333,0.178009,-466.064796,-375.829312
